# NB-13 — Identical objectives across backbones, single-sensor specialists, optical-side consistency (WP9, WP10, WP20)

**Questions** (reviewers R2.3, R4.3, R6.6 for A; R1 and R2.3 for B; R1 and R3.1 for C)

| Group | Runs (× 3 seeds) | Answers |
|---|---|---|
| **A_backbones** (6 runs) | `unet_full`, `manet_full`: U-Net and MANet (MobileNetV2 encoders) with the **same curriculum + SAR-only consistency as SegFormer** | Table 2 mixed architecture with training objective: U-Net and MANet had curriculum only (consistency weight 0). With identical objectives, which backbone is better? |
| **B_specialists** (9 runs) | `seg_spec_s1` (SegFormer trained only on SAR-only input, selected on **SAR-only** val IoU), `seg_spec_s2` (same for optical-only), `seg_full_sel_s1` (the general recipe, but selected on SAR-only val IoU) | Does one general model keep the performance of a specialist? `seg_full_sel_s1` separates the effect of *training* from the effect of *model selection*. |
| **C_symmetric** (3 runs) | `seg_symcons`: the full recipe plus an **optical-only** consistency term | The paper's consistency is one-sided (only the SAR-only view is pulled toward the fused prediction). Does the symmetric version change anything? |
| *D_backbone_matched* (6 runs, optional, off by default) | `unet_matched`, `manet_matched`: fixed dropout at the curriculum's cumulative exposure | Is the "schedule is not needed" result (NB-12, SegFormer) also true on the other backbones? |

All runs: **Study-1 protocol** (Focal-Tversky, AdamW 3e-4, wd 1e-4, 45 epochs, batch 8, EMA 0.99, grad-clip 1, consistency λ = 0.5 where used, seeds 42 / 1 / 2). Selection = mean val IoU over the 3 regimes, except for the specialists and `seg_full_sel_s1` (stated in each name).
Every run logs measured regime shares, per-chip confusion counts on val / test / Bolivia, config, history, checkpoint.

**Rules fixed before looking at the results** (so the write-up cannot drift):
1. A backbone is called "better" than another only if its mean is higher on test SAR-only **and** the difference has the same sign in all 3 seeds **and** exceeds 0.03. Otherwise the paper says "comparable".
2. The general model is said to "retain" a specialist's performance if its regime IoU is within 0.02 of the specialist's and the gap is not larger than the seed spread.
3. Bolivia (15 chips, one event) is reported but not used to choose the headline recipe or backbone.

**Attach** (Add Input):
- `lgmf_v8` (dataset) · Sen1Floods11 · `lgmf-index`
- the **NB-12 notebook output** (gives `seg_full_ref`, `seg_consonly`, `seg_fixed_asym`, … as same-protocol references)
- the **NB-11 audit notebook output** (gives the earlier `unet_drop`, `manet_drop`, `unet_naive`, `segformer_*` numbers)
- on later commits: the previous commit of **this** notebook (RESTORE copies finished NB-13 runs forward)

**Settings:** GPU T4 · Internet **On** (ImageNet encoder weights) · Save & Run All (Commit).
**Budget:** unknown for MANet (48.9 M parameters); SegFormer runs took about 11 min each in NB-12. The loop measures the time per architecture and stops by itself before the session limit; finished runs are never repeated.

## 0. Bootstrap

In [ ]:
import sys, os, glob, json, time, shutil, warnings, re, importlib, dataclasses, copy
warnings.filterwarnings("ignore")
T_START = time.time()
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py",
            "losses.py", "ema.py", "cloud.py", "ramdata.py", "audit.py", "runner.py", "selftest_v8.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_cands = []
print("every lgmf/ folder found under /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)):
    d = os.path.dirname(p); miss = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not miss else 'missing ' + ','.join(miss)}  {d}")
    if not miss: _cands.append((_ver(p), d))
assert _cands, "attach the lgmf_v8 dataset (v7 and older do not have runner/ramdata/cloud/audit)"
_best = max(_cands); assert _best[0] >= (8,), f"best complete lgmf is v{_best[0]}; attach lgmf_v8"
PKG_DIR = _best[1]; sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
for mod, pipname in (("segmentation_models_pytorch", "segmentation-models-pytorch"),):
    try: __import__(mod)
    except ImportError: os.system(f"pip install -q {pipname}")
import numpy as np, pandas as pd, torch
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR, lgmf.__file__
from lgmf.config import LGMFConfig
from lgmf.curriculum import regime_probs
from lgmf.ramdata import RamDataset
from lgmf.runner import run_experiment, is_done, run_dir
from lgmf.audit import metrics_from_counts
print("lgmf", lgmf.__version__, "|", PKG_DIR)

ns = sorted(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)); assert ns, "attach lgmf-index"
idx = os.path.dirname(ns[0]); stats = json.load(open(ns[0]))
for s in ("train", "val", "test", "bolivia"): assert os.path.exists(f"{idx}/{s}_index.json"), s

# --- the index files store absolute image paths from the session that built them. If this session mounts
# --- Sen1Floods11 somewhere else, re-point every path by file name (names are unique) into /kaggle/working/index_fixed.
def _paths_ok(d):
    it = json.load(open(f"{d}/train_index.json"))[0]
    return all(os.path.exists(it[k]) for k in ("s1", "s2", "label") if it.get(k))
if not _paths_ok(idx):
    hits = glob.glob("/kaggle/input/**/HandLabeled", recursive=True)
    assert hits, "index paths are invalid and no 'HandLabeled' folder was found: attach the Sen1Floods11 dataset"
    where = {f: os.path.join(r, f) for r, _, fs in os.walk(hits[0]) for f in fs if f.endswith(".tif")}
    FIX = "/kaggle/working/index_fixed"; os.makedirs(FIX, exist_ok=True); lost = []
    for s in ("train", "val", "test", "bolivia"):
        items = json.load(open(f"{idx}/{s}_index.json"))
        for it in items:
            for k in ("s1", "s2", "label", "jrc"):
                if it.get(k):
                    nm = os.path.basename(it[k])
                    if nm in where: it[k] = where[nm]
                    else: lost.append((s, k, nm))
        json.dump(items, open(f"{FIX}/{s}_index.json", "w"))
    assert not lost, f"{len(lost)} image files not found, e.g. {lost[:3]}"
    print(f"index paths re-pointed to {hits[0]}  ->  {FIX}"); idx = FIX
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU: only for the smoke test)")

# ------------------------------ the knobs ------------------------------
EPOCHS = int(os.environ.get("NB13_EPOCHS", 45))        # never change mid-stream: results live in runs_e<EPOCHS>
SEEDS = [42, 1, 2]
GROUPS_TO_RUN = ["A_backbones", "B_specialists", "C_symmetric"]    # add "D_backbone_matched" only if you want the optional group
TIME_BUDGET_H = 10.5                                    # stop launching runs that would not finish inside this
OUT = "/kaggle/working"
MY_IDS = {"unet_full", "manet_full", "seg_spec_s1", "seg_spec_s2", "seg_full_sel_s1", "seg_symcons", "unet_matched", "manet_matched"}

## 1. Preflight (about 1 min): self-test of the package on synthetic data

In [ ]:
from lgmf import selftest_v8
selftest_v8.run(device=device)       # raises if anything in the v8 package is broken -- better now than after 2 hours

## 2. RESTORE — carry finished NB-13 runs forward from a previous commit of this notebook
Only this notebook's own run ids are copied; the NB-12 / NB-11 inputs are read as references, never copied.

In [ ]:
def restore_previous():
    n = 0
    for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True):
        rid = p.split(os.sep)[-3]
        if rid not in MY_IDS: continue
        src = os.path.dirname(p); rel = src.split(f"runs_e{EPOCHS}" + os.sep, 1)[1]
        dst = os.path.join(OUT, f"runs_e{EPOCHS}", rel)
        if not os.path.exists(os.path.join(dst, "result.json")):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst); n += 1
    return n
print("restored", restore_previous(), "finished NB-13 run(s) from attached inputs")
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")): print("  have", p.split(f"runs_e{EPOCHS}/")[1].rsplit("/result.json", 1)[0])

## 3. Data into RAM (once) and the run table

In [ ]:
base = LGMFConfig(); base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]; base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]
t0 = time.time()
TRAIN = RamDataset(base, f"{idx}/train_index.json", augment=True)
EVAL = {s: RamDataset(base, f"{idx}/{s}_index.json", augment=False) for s in ("val", "test", "bolivia")}
print({"train": len(TRAIN), **{k: len(v) for k, v in EVAL.items()}}, f"cached in {time.time()-t0:.0f}s")
assert len(TRAIN) == 250 and len(EVAL["val"]) == 89 and len(EVAL["test"]) == 90 and len(EVAL["bolivia"]) == 15, "split sizes differ from the paper"

def make_cfg(overrides, seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size, c.ema_decay = EPOCHS, 3e-4, 8, 0.99
    c.loss_type, c.consistency_weight, c.num_workers = "focal_tversky", 0.5, 0
    c.s1_mean, c.s1_std, c.s2_mean, c.s2_std = stats["s1_mean"], stats["s1_std"], stats["s2_mean"], stats["s2_std"]
    c.seed = seed
    for k, v in overrides.items(): setattr(c, k, v)
    return c

_c = make_cfg({}, 0)
MATCHED = tuple(float(x) for x in np.mean([regime_probs(_c, e) for e in range(EPOCHS)], 0))
print("curriculum cumulative exposure (both, SAR-only, optical-only):", np.round(MATCHED, 4))

SEG   = dict(kind="concat", arch="segformer", encoder="mit_b0")
UNET  = dict(kind="concat", arch="unet",  encoder="mobilenet_v2")
MANET = dict(kind="concat", arch="manet", encoder="mobilenet_v2")
ONLY_S1 = dict(use_curriculum=False, fixed_regime_probs=(0.0, 1.0, 0.0), consistency_weight=0.0)   # every example is SAR-only; no fused target exists, so no consistency
ONLY_S2 = dict(use_curriculum=False, fixed_regime_probs=(0.0, 0.0, 1.0), consistency_weight=0.0)
MATCHED_FIX = dict(use_curriculum=False, fixed_regime_probs=MATCHED)
RUNS = [   # (run_id, group, model spec, config overrides, selection criterion)
    ("unet_full",       "A_backbones",        UNET,  {},                                    "mean"),
    ("manet_full",      "A_backbones",        MANET, {},                                    "mean"),
    ("seg_spec_s1",     "B_specialists",      SEG,   ONLY_S1,                               "s1_only"),
    ("seg_spec_s2",     "B_specialists",      SEG,   ONLY_S2,                               "s2_only"),
    ("seg_full_sel_s1", "B_specialists",      SEG,   {},                                    "s1_only"),
    ("seg_symcons",     "C_symmetric",        SEG,   dict(consistency_symmetric=True),      "mean"),
    ("unet_matched",    "D_backbone_matched", UNET,  MATCHED_FIX,                           "mean"),
    ("manet_matched",   "D_backbone_matched", MANET, MATCHED_FIX,                           "mean"),
]
assert {r[0] for r in RUNS} == MY_IDS
display(pd.DataFrame([dict(run=r, group=g, arch=s["arch"], encoder=s["encoder"], curriculum=o.get("use_curriculum", True),
                           fixed=o.get("fixed_regime_probs"), consistency=o.get("consistency_weight", 0.5),
                           symmetric=o.get("consistency_symmetric", False), select=sel) for r, g, s, o, sel in RUNS]))

## 4. Train (seed-major; resumable; the time estimate is kept per architecture; stops before the session limit)

In [ ]:
todo = [(rid, g, spec, ov, sel, sd) for sd in SEEDS for rid, g, spec, ov, sel in RUNS if g in GROUPS_TO_RUN and not is_done(OUT, EPOCHS, rid, sd)]
print(f"{len(todo)} run(s) to do in {GROUPS_TO_RUN}")
dur = {}                                   # architecture -> minutes per finished run
for rid, g, spec, ov, sel, sd in todo:
    seen = [x for v in dur.values() for x in v]
    est = (np.mean(dur[spec["arch"]]) if spec["arch"] in dur else (max(seen) * 1.5 if seen else 35.0)) * 1.15
    left = TIME_BUDGET_H * 60 - (time.time() - T_START) / 60
    if est > left:
        print(f"STOP: next run ({rid}, seed {sd}) needs ~{est:.0f} min, only {left:.0f} min of budget left. "
              f"Save this version, attach its output to a new commit, and run again (finished runs are skipped).")
        break
    t1 = time.time()
    run_experiment(rid, make_cfg(ov, sd), spec, TRAIN, EVAL["val"], EVAL, OUT, device, select=sel)
    dur.setdefault(spec["arch"], []).append((time.time() - t1) / 60)
print("minutes per run by architecture:", {k: round(float(np.mean(v)), 1) for k, v in dur.items()})
print(f"done; {sum(len(v) for v in dur.values())} run(s) this session, {(time.time()-T_START)/3600:.1f} h elapsed")

## 5. Results — every run, test and Bolivia, mean ± std over seeds (ddof = 1)
Row prefixes: no prefix = trained in this notebook · `nb12:` = NB-12 run (same protocol, v8 pipeline) · `ref11:` = the earlier runs behind the paper's Table 2 (from the NB-11 audit).
`ref11:unet_drop` / `ref11:manet_drop` had **consistency weight 0** (curriculum only); `unet_full` / `manet_full` add the consistency term, which is the identical-objective comparison.

In [ ]:
recs, seen_ref = [], set()
def add_result(r, prefix=""):
    for split in ("val", "test", "bolivia"):
        for reg, m in r[split].items():
            recs.append(dict(run=prefix + r["run_id"], seed=r["seed"], split=split, regime=reg, iou=m["iou"], flood_iou=m["flood_iou"]))
for p in glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json"):
    add_result(json.load(open(p)))
# NB-12 references: any other run found in the attached inputs (never copied)
for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True):
    r = json.load(open(p))
    if r["run_id"] in MY_IDS or (r["run_id"], r["seed"]) in seen_ref: continue
    seen_ref.add((r["run_id"], r["seed"])); add_result(r, "nb12:")
print("NB-12 reference runs found:", sorted({k[0] for k in seen_ref}) or "NONE (attach the NB-12 notebook output)")
# NB-11 references (the runs behind the paper's Table 2)
ref_files = glob.glob("/kaggle/input/**/per_chip_counts.csv.gz", recursive=True)
if ref_files:
    pc = pd.read_csv(ref_files[0]); pc = pc[(pc.variant == "zero") & pc.config.isin(["segformer_concat", "segformer_noconsist", "unet_drop", "manet_drop", "unet_naive"])]
    g = pc.groupby(["config", "seed", "split", "regime"])[["tp", "fp", "fn", "tn", "ftp", "ffp", "ffn", "ftn"]].sum().reset_index()
    g["iou"] = metrics_from_counts(g.tp, g.fp, g.fn, g.tn)["iou"]; g["flood_iou"] = metrics_from_counts(g.ftp, g.ffp, g.ffn, g.ftn)["iou"]
    for r in g.itertuples(): recs.append(dict(run="ref11:" + r.config, seed=int(r.seed), split=r.split, regime=r.regime, iou=r.iou, flood_iou=r.flood_iou))
    print("NB-11 references loaded from", ref_files[0])
else:
    print("NB-11 output not attached: no ref11:* rows")
R = pd.DataFrame(recs); R.to_csv(f"{OUT}/nb13_per_seed.csv", index=False)

ORDER = ["ref11:unet_naive", "ref11:unet_drop", "unet_full", "unet_matched", "ref11:manet_drop", "manet_full", "manet_matched",
         "ref11:segformer_noconsist", "ref11:segformer_concat", "nb12:seg_full_ref", "nb12:seg_consonly", "nb12:seg_fixed_asym", "nb12:seg_matched",
         "seg_full_sel_s1", "seg_spec_s1", "seg_spec_s2", "seg_symcons"]
def table(split):
    g = R[R.split == split].groupby(["run", "regime"]).iou.agg(["mean", "std", "count"]).reset_index()
    g["cell"] = g.apply(lambda r: f"{r['mean']:.3f}±{(r['std'] if r['std'] == r['std'] else 0):.3f}", axis=1)
    t = g.pivot(index="run", columns="regime", values="cell")[["both", "s1_only", "s2_only"]]
    t["n"] = g.groupby("run")["count"].max()
    return t.reindex([x for x in ORDER if x in t.index] + [x for x in t.index if x not in ORDER])
pd.set_option("display.width", 250)
for sp in ("test", "bolivia"):
    print(f"\n=== {sp.upper()}  (IoU, mean ± std over seeds, ddof=1) ==="); print(table(sp).to_string())
table("test").to_csv(f"{OUT}/nb13_table_test.csv"); table("bolivia").to_csv(f"{OUT}/nb13_table_bolivia.csv")

## 6. Paired contrasts (same seed), grouped by question
Differences are computed seed by seed, then summarised. With 3 seeds this is **descriptive**: read the per-seed column; `same` = the sign is identical in all seeds.
Applying the rules at the top: "better" needs mean > 0.03 **and** same sign in all seeds; below that the paper says "comparable".

In [ ]:
def per_seed(run, split, regime):
    return R[(R.run == run) & (R.split == split) & (R.regime == regime)].set_index("seed")["iou"]
CONTRASTS = [   # (question, label, A, B) -> A minus B
    ("WP9 identical objectives", "consistency added to U-Net (curriculum fixed)",   "unet_full",            "ref11:unet_drop"),
    ("WP9 identical objectives", "consistency added to MANet (curriculum fixed)",  "manet_full",           "ref11:manet_drop"),
    ("WP9 identical objectives", "consistency added to SegFormer (for comparison)", "ref11:segformer_concat", "ref11:segformer_noconsist"),
    ("WP9 identical objectives", "dropout curriculum alone on U-Net (vs both-only training)", "ref11:unet_drop", "ref11:unet_naive"),
    ("WP9 identical objectives", "U-Net minus SegFormer, same full recipe",         "unet_full",            "nb12:seg_full_ref"),
    ("WP9 identical objectives", "MANet minus SegFormer, same full recipe",         "manet_full",           "nb12:seg_full_ref"),
    ("WP9 identical objectives", "U-Net minus MANet, same full recipe",             "unet_full",            "manet_full"),
    ("WP9 optional", "U-Net: matched fixed mix minus curriculum",                   "unet_matched",         "unet_full"),
    ("WP9 optional", "MANet: matched fixed mix minus curriculum",                   "manet_matched",        "manet_full"),
    ("WP10 specialists", "SAR: general (mean-selected) minus SAR specialist",       "nb12:seg_full_ref",    "seg_spec_s1"),
    ("WP10 specialists", "SAR: general (SAR-selected) minus SAR specialist",        "seg_full_sel_s1",      "seg_spec_s1"),
    ("WP10 specialists", "SAR: effect of selecting on SAR-only val IoU",            "seg_full_sel_s1",      "nb12:seg_full_ref"),
    ("WP10 specialists", "optical: general (mean-selected) minus optical specialist", "nb12:seg_full_ref",  "seg_spec_s2"),
    ("WP20 asymmetry", "optical-side term added (symmetric) minus default",         "seg_symcons",          "nb12:seg_full_ref"),
]
rows, lines = [], []
for q, lab, A, B in CONTRASTS:
    out = []
    for reg in ("s1_only", "s2_only", "both"):
        seg = []
        for sp in ("test", "bolivia"):
            a, b = per_seed(A, sp, reg), per_seed(B, sp, reg); common = a.index.intersection(b.index)
            if len(common) == 0: seg.append(f"{sp}: n/a"); continue
            d = (a[common] - b[common])
            rows.append(dict(question=q, contrast=lab, split=sp, regime=reg, n=len(common), mean_diff=d.mean(), seed_sd=d.std() if len(d) > 1 else np.nan,
                             per_seed=" ".join(f"{s}:{v:+.3f}" for s, v in d.items()), same_sign=bool((d > 0).all() or (d < 0).all())))
            seg.append(f"{sp}: {d.mean():+.3f} [{' '.join(f'{s}:{v:+.3f}' for s, v in d.items())}]{' same' if ((d > 0).all() or (d < 0).all()) else ''}")
        out.append(f"      {reg:8s} " + "   ".join(seg))
    lines.append((q, lab, out))
C = pd.DataFrame(rows); C.to_csv(f"{OUT}/nb13_contrasts.csv", index=False)
last = None
for q, lab, out in lines:
    if q != last: print(f"\n##### {q}"); last = q
    print(f"  {lab}"); print("\n".join(out))

## 7. Does one general model keep a specialist's performance? (WP10)
Specialists are trained **only** on their own input and selected on that regime's validation IoU. `seg_full_sel_s1` is the general recipe selected the same way, so (general, SAR-selected) vs (specialist) isolates *training*, and (SAR-selected) vs (mean-selected) isolates *selection*.

In [ ]:
def ms(run, split, reg):
    v = R[(R.run == run) & (R.split == split) & (R.regime == reg)].iou.values
    return (v.mean(), v.std(ddof=1) if len(v) > 1 else np.nan, len(v)) if len(v) else (np.nan, np.nan, 0)
rows = []
for reg, spec_run in (("s1_only", "seg_spec_s1"), ("s2_only", "seg_spec_s2")):
    for sp in ("test", "bolivia"):
        sm = ms(spec_run, sp, reg)
        for lab, run in (("general, mean-selected (nb12:seg_full_ref)", "nb12:seg_full_ref"), ("general, SAR-selected (seg_full_sel_s1)", "seg_full_sel_s1"),
                         ("general, paper run (ref11:segformer_concat)", "ref11:segformer_concat")):
            if reg == "s2_only" and run == "seg_full_sel_s1": continue
            gm = ms(run, sp, reg)
            if gm[2] == 0 or sm[2] == 0: continue
            rows.append(dict(regime=reg, split=sp, specialist=f"{sm[0]:.3f}±{sm[1]:.3f}", general=lab, general_iou=f"{gm[0]:.3f}±{gm[1]:.3f}",
                             gap=round(gm[0] - sm[0], 3), retained_pct=round(100 * gm[0] / sm[0], 1) if sm[0] > 0 else np.nan))
S = pd.DataFrame(rows)
print(S.to_string(index=False) if len(S) else "no specialist results yet")
S.to_csv(f"{OUT}/nb13_specialist_table.csv", index=False)
print("\nrule 2: 'retains' needs |gap| <= 0.02 and the gap not larger than the seed spread")

## 8. Backbone summary under identical objectives (WP9) — with parameter counts

In [ ]:
PARAMS = {"segformer": 3.72, "unet": 6.63, "manet": 48.89}          # millions; confirmed against result.json below where available
chk = {}
for p in glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json"):
    r = json.load(open(p)); chk[r["spec"]["arch"]] = r["params_M"]
print("parameter counts read from this notebook's runs (M):", {k: round(v, 2) for k, v in chk.items()})
rows = []
for lab, run, arch, cons in (("U-Net, curriculum only (paper)", "ref11:unet_drop", "unet", "no"), ("U-Net, curriculum + consistency", "unet_full", "unet", "yes"),
                             ("MANet, curriculum only (paper)", "ref11:manet_drop", "manet", "no"), ("MANet, curriculum + consistency", "manet_full", "manet", "yes"),
                             ("SegFormer, curriculum only (paper)", "ref11:segformer_noconsist", "segformer", "no"), ("SegFormer, curriculum + consistency (NB-12 re-run)", "nb12:seg_full_ref", "segformer", "yes")):
    d = dict(model=lab, params_M=PARAMS[arch], consistency=cons)
    for sp in ("test", "bolivia"):
        for reg in ("both", "s1_only", "s2_only"):
            m, s, n = ms(run, sp, reg); d[f"{sp}_{reg}"] = f"{m:.3f}±{s:.3f}" if n else "n/a"
    rows.append(d)
B = pd.DataFrame(rows); print(B.to_string(index=False)); B.to_csv(f"{OUT}/nb13_backbone_summary.csv", index=False)

## 9. Measured regime exposure of this notebook's runs (counted, not computed)

In [ ]:
rows = []
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")):
    r = json.load(open(p))
    rows.append(dict(run=r["run_id"], seed=r["seed"], measured_both=r["measured_cumulative_shares"][0], measured_sar=r["measured_cumulative_shares"][1],
                     measured_opt=r["measured_cumulative_shares"][2], scheduled_both=r["scheduled_cumulative_probs"][0],
                     scheduled_sar=r["scheduled_cumulative_probs"][1], scheduled_opt=r["scheduled_cumulative_probs"][2], best_epoch=r["best_epoch"],
                     params_M=r["params_M"], train_min=r["train_minutes"]))
X = pd.DataFrame(rows)
if len(X):
    g = X.groupby("run").mean(numeric_only=True).drop(columns="seed")
    print(g.round(4).to_string()); g.to_csv(f"{OUT}/nb13_measured_exposure.csv")

## 10. What to send back
When the commit has finished: send me the printed output of sections 5–9 (or the executed `.ipynb`).
If the loop printed `STOP: …`, save the version, attach its output to a new commit of this notebook and run again.
Optional group D (`GROUPS_TO_RUN = [..., "D_backbone_matched"]`) only if sections 5–6 show a backbone difference worth explaining.